# 03 — Churn Analysis

Churn proxy definition, churn-risk modeling and top-risk targeting metrics.

**Project:** Marketing Analytics Causal & LTV Lab  
**Phase:** Phase 1 — Customer Analytics, Retention, Churn and LTV Baseline

> This notebook is designed as a hands-on learning notebook. Run each section, inspect the output, and discuss the interpretation before moving to the next step.


## 1. Notebook objective

This notebook creates a churn proxy and builds churn-risk models.

Because there is no explicit churn label, we define churn using recency:

`churn_90d = Last_Transaction_Days_Ago > 90`

This is a practical proxy, not a ground-truth churn event.


In [ ]:
# Core imports
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix
)
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_RAW = Path("../data/raw/digital_wallet_ltv_dataset.csv")
DATA_PROCESSED = Path("../data/processed")
REPORTS = Path("../reports")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PROCESSED / 'wallet_retention_features.csv') if (DATA_PROCESSED / 'wallet_retention_features.csv').exists() else pd.read_csv(DATA_RAW)
df.head()


## 2. Create churn target


In [ ]:
df["churn_90d"] = (df["Last_Transaction_Days_Ago"] > 90).astype(int)

print(df["churn_90d"].value_counts(normalize=True).rename("rate"))
print(df["churn_90d"].value_counts())

plt.figure(figsize=(5, 4))
df["churn_90d"].value_counts().sort_index().plot(kind="bar")
plt.title("Churn Proxy Distribution")
plt.xlabel("churn_90d")
plt.ylabel("Customer count")
plt.show()


## 3. Feature set design


In [ ]:
target = "churn_90d"

drop_cols = [
    "Customer_ID",
    "LTV",
    "churn_90d",
    "retained_30d",
    "dormant_90d",
    "Last_Transaction_Days_Ago"  # direct target leakage
]

candidate_features = [c for c in df.columns if c not in drop_cols]

X = df[candidate_features]
y = df[target]

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric:", num_features)
print("Categorical:", cat_features)


## 4. Train/test split and preprocessing


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_features),
    ("cat", categorical_pipe, cat_features)
])


## 5. Baseline and model comparison


In [ ]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    results.append({
        "model": name,
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
        "positive_rate_predicted": pred.mean()
    })

    print(f"\n=== {name} ===")
    print(classification_report(y_test, pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, pred))

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
display(results_df)


## 6. Business metric: recall at top risk decile


In [ ]:
best_name = results_df.iloc[0]["model"]
best_model = models[best_name]

best_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", best_model)
])
best_pipe.fit(X_train, y_train)

df_scored = df.copy()
df_scored["churn_risk_score"] = best_pipe.predict_proba(X)[:, 1]
df_scored["risk_decile"] = pd.qcut(
    df_scored["churn_risk_score"].rank(method="first"),
    q=10,
    labels=False
) + 1

top_decile = df_scored[df_scored["risk_decile"] == 10]
recall_top_decile = top_decile["churn_90d"].sum() / df_scored["churn_90d"].sum()

print("Best model:", best_name)
print("Churn recall captured in top risk decile:", round(recall_top_decile, 3))
display(
    df_scored.groupby("risk_decile")
    .agg(customers=("Customer_ID", "count"), churn_rate=("churn_90d", "mean"), avg_ltv=("LTV", "mean"))
)


## 7. Save churn features


In [ ]:
output = DATA_PROCESSED / "wallet_churn_features.csv"
df_scored.to_csv(output, index=False)
print(f"Saved: {output}")


## Discussion prompts

1. Why do we remove `Last_Transaction_Days_Ago` from features?
2. Why is PR-AUC important when churn is imbalanced?
3. Why is top-decile recall more business-friendly than accuracy?
4. Which high-risk users should we target later?
